In [2]:
!pip install sentence-transformers -q



In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [4]:
# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample sentences
sentences = [
    "I love playing football",
    "Soccer is my favorite sport",
    "Machine learning is fascinating",
    "Deep learning uses neural networks",
    "I enjoy cooking Italian food",
    "Pasta and pizza are delicious"
]

# Generate embeddings
embeddings = model.encode(sentences)

print(f"Shape: {embeddings.shape}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (6, 384)


In [5]:
def cosine_similarity(vec_a, vec_b):
    dot_product = np.dot(vec_a, vec_b)
    magnitude_a = np.linalg.norm(vec_a)
    magnitude_b = np.linalg.norm(vec_b)
    return dot_product / (magnitude_a * magnitude_b)

# Compare pairs
pairs = [
    ("I love playing football", "Soccer is my favorite sport"),   # should be HIGH
    ("I love playing football", "Machine learning is fascinating"), # should be LOW
    ("Deep learning uses neural networks", "Machine learning is fascinating"), # should be HIGH
]

for s1, s2 in pairs:
    v1 = model.encode(s1)
    v2 = model.encode(s2)
    score = cosine_similarity(v1, v2)
    print(f"{score:.4f} | {s1[:30]}... ↔ {s2[:30]}...")

0.6545 | I love playing football... ↔ Soccer is my favorite sport...
0.1213 | I love playing football... ↔ Machine learning is fascinatin...
0.4898 | Deep learning uses neural netw... ↔ Machine learning is fascinatin...


In [6]:
def similarity_matrix(embeddings):
    n = len(embeddings)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            matrix[i][j] = cosine_similarity(embeddings[i], embeddings[j])
    return matrix

sim_matrix = similarity_matrix(embeddings)

# Pretty print
print(f"{'':>35}", end="")
for s in sentences:
    print(f"{s[:8]:>10}", end="")
print()

for i, s in enumerate(sentences):
    print(f"{s[:35]:>35}", end="")
    for j in range(len(sentences)):
        print(f"{sim_matrix[i][j]:>10.3f}", end="")
    print()

                                     I love p  Soccer i  Machine   Deep lea  I enjoy   Pasta an
            I love playing football     1.000     0.655     0.121     0.047     0.341     0.131
        Soccer is my favorite sport     0.655     1.000     0.220     0.026     0.362     0.222
    Machine learning is fascinating     0.121     0.220     1.000     0.490     0.161     0.126
 Deep learning uses neural networks     0.047     0.026     0.490     1.000     0.067     0.051
       I enjoy cooking Italian food     0.341     0.362     0.161     0.067     1.000     0.604
      Pasta and pizza are delicious     0.131     0.222     0.126     0.051     0.604     1.000


In [7]:
documents = [
    "Python is a popular programming language for data science",
    "Neural networks are inspired by the human brain",
    "Gradient descent optimizes model parameters",
    "Pandas is used for data manipulation in Python",
    "Transformers revolutionized natural language processing",
    "Random forests combine multiple decision trees",
    "BERT is a bidirectional transformer model",
    "NumPy provides efficient array operations",
    "Overfitting occurs when a model memorizes training data",
    "Attention mechanism allows models to focus on relevant tokens"
]

# Step 1 — Embed all documents ONCE (offline step)
doc_embeddings = model.encode(documents)
print(f"Document store ready: {doc_embeddings.shape}")
# → (10, 384)

# Step 2 — Semantic search function
def semantic_search(query, documents, doc_embeddings, top_k=3):
    # Embed the query
    query_embedding = model.encode(query)

    # Compute cosine similarity against all docs
    scores = [
        cosine_similarity(query_embedding, doc_emb)
        for doc_emb in doc_embeddings
    ]

    # Rank and return top-k
    ranked = sorted(
        zip(scores, documents),
        key=lambda x: x[0],
        reverse=True
    )

    return ranked[:top_k]

# Step 3 — Run queries
queries = [
    "How do neural networks learn?",
    "What tools help with tabular data in Python?",
    "How does BERT process text?"
]

for query in queries:
    print(f"\n🔍 Query: {query}")
    results = semantic_search(query, documents, doc_embeddings, top_k=3)
    for rank, (score, doc) in enumerate(results, 1):
        print(f"  {rank}. [{score:.4f}] {doc}")

Document store ready: (10, 384)

🔍 Query: How do neural networks learn?
  1. [0.6038] Neural networks are inspired by the human brain
  2. [0.3690] Overfitting occurs when a model memorizes training data
  3. [0.3448] Gradient descent optimizes model parameters

🔍 Query: What tools help with tabular data in Python?
  1. [0.5176] Python is a popular programming language for data science
  2. [0.5030] Pandas is used for data manipulation in Python
  3. [0.2700] NumPy provides efficient array operations

🔍 Query: How does BERT process text?
  1. [0.5102] BERT is a bidirectional transformer model
  2. [0.3955] Attention mechanism allows models to focus on relevant tokens
  3. [0.3167] Transformers revolutionized natural language processing


##Using the Built-in Utility (Production Pattern)

In [8]:
from sentence_transformers import util

query = "Which Python library handles arrays efficiently?"
query_embedding = model.encode(query, convert_to_tensor=True)
doc_embeddings_tensor = model.encode(documents, convert_to_tensor=True)

results = util.semantic_search(query_embedding, doc_embeddings_tensor, top_k=3)

print(f"\n🔍 Query: {query}")
for hit in results[0]:
    print(f"  [{hit['score']:.4f}] {documents[hit['corpus_id']]}")


🔍 Query: Which Python library handles arrays efficiently?
  [0.6685] NumPy provides efficient array operations
  [0.4920] Python is a popular programming language for data science
  [0.4245] Pandas is used for data manipulation in Python
